## dual_clip +  权重反转 为核心

In [ ]:
import  torch
import torchr.nn.functional as F

def sg(x):
    return x.detach() - x + x

# ------ASPO核心：dual-clip + 权重反转------
def aspo_loss(logits_new,logtis_old,action,advantage,eps_hard=0.2,esp_soft=0.05):
    """
    logits_new:[B,L,V] 当前策略
    logits_old:[B,L,V] 旧策略
    advantage:[B,L] 优势
    action : [B,L] token_id
    return loss
    """
    B,L,V = logits_new.shape
    tgt = action.reshape(B * L)
    
    # 1 计算ratio
    #  reduction='none'在 ASPO/GRPO/PPO 实现里，必须用 'none'，否则无法得到 逐 token 的 log π(a_t|s_t)，也就没法计算重要性权重 r_t
    logp_new = -F.cross_entropy(
        logits_new.view(-1，V),tgt, reduction='none'
    ).view(B,L)
    
    logp_old = -F.cross_entropy(
        logtis_old.view(-1,V),tgt,reduction='none'
    ).view(B,L)
    ratio = (logp_new - logp_old).exp()
    
    # 2 硬裁剪
    low,high = 1- eps_hard,1+eps_hard
    clipped = torch.clamp(ratio,low,high)
    surr1 = ratio * advantage
    surr2 = clipped * advantage
    loss_hard = -torch.min(surr1,surr2)
    
    # 3 ASPO 权重反转： 仅对dual-clip区域加软项
    big_mask = ratio > high
    small_mask = ratio < low
    dual_maks = big_mask | small_mask
    
    # 权重反转： (clip - r) 负号自动相反
    """
    ratio 过大 → r > 1+ε
    clipped = 1+ε
    clipped - ratio = (1+ε) - r   < 0      // 负号
    
    ratio 过小 → r < 1−ε
    clipped = 1−ε
    clipped - ratio = (1−ε) - r   > 0      // 正号
    """
     soft_term = sg(ratio) * eps_soft * (clipped - ratio) * advantage
    loss_total = loss_hard + torch.where(dual_mask, soft_term,torch.zeros_like(advantage))

In [ ]:
B, L, V = 2, 10, 50257
    logits_new = torch.randn(B, L, V, requires_grad=True)
    logits_old = torch.randn(B, L, V).detach()
    action = torch.randint(0, V, (B, L))
    adv = torch.randn(B, L)

    loss = aspo_loss(logits_new, logits_old, action, adv)
    loss.backward()
    print("ASPO loss:", loss.item())

##  ASPO 的三个核心模块：
- Token Masking（硬裁剪）
- Asymmetric Importance Sampling（正优势 token 权重翻转）
- Soft Dual-Clipping（防极端值，保留梯度）

代码以「token-level PPO」为骨架，可直接嵌入到 GRPO / PPO 训练循环里。为简化阅读，只保留单条样本（L 个 token）的前向-损失-反向部分；

batch 版本只需在外层再包一层 DataLoader 即可。
